# Physical AI Platform - CFD Backend and GPU Research Notebook

**Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.**  
**AyeAI®, AyeAM® and AyeCNSe® are registered trademarks of Abhishek Choudhary.**  
**Notebook licence: GPL-3.0-only.**  
**Third-party software retains its own copyright and licence.**

- Notebook ID: `pap-cfd-backend-colab`
- Notebook version: `1.0.0`
- Maintainer: Physical AI Platform / AyeAI
- Purpose: reproducible analytical reference, accelerator capability discovery, external CFD executable discovery and provenance-bearing result packaging.
- Validation boundary: this notebook is not a substitute for an independently validated 2D/3D CFD solver or physical experiment.

## Open in Google Colab

After this notebook is committed to the platform repository, Grok must replace the repository placeholders in `notebooks/README.md` with the actual immutable commit URL and publish the `Open in Colab` link. The correct URL form is:

`https://colab.research.google.com/github/<OWNER>/<REPOSITORY>/blob/<COMMIT_SHA>/drop-in/cfd/notebooks/CFD_Backend_Colab.ipynb`

Do not publish a fabricated or branch-ambiguous link. Pin the commit SHA for reproducibility. Colab runtime availability, GPU type, quotas and lifetime are controlled by Google and are not guaranteed.

## What this notebook does

1. Captures runtime and accelerator metadata.
2. Recomputes a closed-form Couette-Poiseuille reference and checks physical invariants.
3. Runs a parameter sweep on GPU with CuPy when a compatible NVIDIA CUDA runtime is available; otherwise runs the same sweep on CPU.
4. Discovers (but does not silently install or claim) external CFD executables such as OpenFOAM, SU2 and Code_Saturne.
5. Writes a provenance manifest suitable for return to the platform's canonical evidence workflow.

GPU acceleration here is used for batched parametric calculations and research workflows. It does **not** claim that OpenFOAM, SU2 or Code_Saturne is automatically running its CFD solver on the GPU.

## 0. Runtime selection

In Colab choose **Runtime → Change runtime type → Hardware accelerator → GPU** when GPU computation is intended. A GPU runtime does not imply a GPU is present or used. The next cells inspect actual runtime capabilities. CPU execution remains a supported fallback.

In [ ]:
import os, sys, platform, json, shutil, subprocess, hashlib, time, math
from datetime import datetime, timezone

runtime = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "machine": platform.machine(),
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "colab": bool(os.environ.get("COLAB_RELEASE_TAG")),
    "nvidia_smi": shutil.which("nvidia-smi"),
}
if runtime["nvidia_smi"]:
    try:
        runtime["nvidia_smi_output"] = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
            check=True, capture_output=True, text=True, timeout=15
        ).stdout.strip()
    except Exception as exc:
        runtime["nvidia_smi_error"] = repr(exc)
print(json.dumps(runtime, indent=2))

## 1. Dependencies

The analytical reference and CPU sweep require NumPy. Matplotlib is used for plots. CuPy is optional and is installed only after an NVIDIA device and CUDA runtime are detected. This avoids installing a CUDA wheel that does not match the active runtime.

In [ ]:
%pip -q install numpy matplotlib

In [ ]:
# Install the CUDA 12 CuPy wheel only when an NVIDIA runtime is visible.
# No GPU package is installed on CPU-only runtimes. A failed optional install does not abort the notebook.
if shutil.which('nvidia-smi'):
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=15)
    cuda_line = next((line for line in smi.stdout.splitlines() if 'CUDA Version:' in line), '')
    if smi.returncode == 0 and 'CUDA Version: 12.' in cuda_line:
        install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'cupy-cuda12x'], capture_output=True, text=True, timeout=180)
        if install.returncode != 0:
            print('Optional CuPy install failed; CPU fallback remains available:', install.stderr[-1000:])
    else:
        print('GPU detected but CUDA wheel not auto-selected; inspect runtime and install a matching CuPy build manually.')
else:
    print('CPU-only runtime: CuPy install skipped.')


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

GPU = {"available": False, "backend": "NumPy/CPU", "reason": "No compatible CuPy runtime detected"}
try:
    import cupy as cp
    _ = cp.cuda.runtime.getDeviceCount()
    GPU = {"available": True, "backend": "CuPy", "device_count": int(cp.cuda.runtime.getDeviceCount()),
           "device": cp.cuda.runtime.getDeviceProperties(0)["name"].decode(errors="replace")}
except Exception as exc:
    cp = None
    GPU["reason"] = str(exc)
print(json.dumps(GPU, indent=2))

## 2. Analytical channel-flow verification

For steady, fully developed, incompressible Newtonian plane-channel flow with constant properties and constant streamwise pressure gradient:

`u(y) = U0 + (U1-U0)y/H - (dp/dx)y(H-y)/(2μ)`

This is an analytical reference for the package's 1D laminar channel solver. It does not validate turbulence, transient flow, 2D/3D geometry or an external solver.

In [ ]:
def couette_poiseuille(y, H, mu, dpdx, U0=0.0, U1=0.0):
    vals = [H, mu, dpdx, U0, U1]
    if not all(np.isfinite(v) for v in vals) or H <= 0 or mu <= 0:
        raise ValueError("H and mu must be positive; all inputs must be finite")
    y = np.asarray(y, dtype=float)
    if np.any(~np.isfinite(y)) or np.any(y < 0) or np.any(y > H):
        raise ValueError("y must be finite and lie in [0,H]")
    return U0 + (U1-U0)*y/H - dpdx*y*(H-y)/(2*mu)

H, mu, dpdx, U0, U1 = 0.01, 1.002e-3, -0.1, 0.0, 0.0
y = np.linspace(0.0, H, 257)
u = couette_poiseuille(y, H, mu, dpdx, U0, U1)
assert np.isclose(u[0], U0, rtol=0, atol=1e-14)
assert np.isclose(u[-1], U1, rtol=0, atol=1e-14)
assert np.all(np.isfinite(u))
q_exact = (U0+U1)*H/2 - dpdx*H**3/(12*mu)
q_trap = np.trapezoid(u, y)
assert abs(q_trap-q_exact) / max(abs(q_exact), 1e-30) < 2e-5
print({"max_velocity_m_s": float(np.max(np.abs(u))), "flow_rate_per_span_m2_s": float(q_exact),
       "trapezoid_relative_error": float(abs(q_trap-q_exact)/abs(q_exact))})
plt.plot(u, y); plt.xlabel("u [m/s]"); plt.ylabel("y [m]"); plt.title("Analytical Couette-Poiseuille profile"); plt.grid(True); plt.show()

## 3. Batched parameter sweep - GPU when actually available

The sweep evaluates the same analytical relation across a parameter ensemble. CuPy is used only if its CUDA runtime successfully enumerates a device. This demonstrates an accelerator path for parametric research, post-processing and surrogate-data generation. It is **not** a GPU implementation of a general Navier-Stokes solver.

In [ ]:
Array = cp if GPU["available"] else np
n = 250_000
heights = Array.linspace(0.004, 0.02, n)
viscosities = Array.linspace(0.7e-3, 1.5e-3, n)
pressure_gradients = Array.linspace(-2.0, -0.05, n)
# Plane Poiseuille flow, stationary walls: q per unit span = -dp/dx * H^3 / (12 mu)
flow_rates = -pressure_gradients * heights**3 / (12.0 * viscosities)
max_speeds = -pressure_gradients * heights**2 / (8.0 * viscosities)
if GPU["available"]:
    Array.cuda.Stream.null.synchronize()
summary = {
    "array_backend": GPU["backend"], "samples": n,
    "flow_rate_min": float(Array.min(flow_rates).get() if GPU["available"] else Array.min(flow_rates)),
    "flow_rate_max": float(Array.max(flow_rates).get() if GPU["available"] else Array.max(flow_rates)),
    "max_speed_max": float(Array.max(max_speeds).get() if GPU["available"] else Array.max(max_speeds)),
}
print(json.dumps(summary, indent=2))

## 4. External CFD backend discovery

This cell discovers installed executables only. It does not imply a backend is integrated or compatible. Full solver invocation requires a version-pinned adapter, validated case writer, mesh/field mapping, parser, process/resource controls and benchmark acceptance. A binary name alone is insufficient.

In [ ]:
executables = ["foamVersion", "foamRun", "simpleFoam", "icoFoam", "SU2_CFD", "code_saturne", "cs"]
backend_probe = {}
for exe in executables:
    path = shutil.which(exe)
    backend_probe[exe] = {"available": bool(path), "path": path}
print(json.dumps(backend_probe, indent=2))

## 5. Optional activation of an external CFD engine

The following opt-in cell can install the OpenCFD v2606 package in a Colab Linux runtime using the official Debian/Ubuntu repository helper. It is disabled by default because it changes the runtime, downloads native packages and consumes disk/time. Read the official instructions before setting the flag. The cell records the installed version; it does not claim GPU execution.

The package name is version-pinned to `openfoam2606-default`. If the official repository does not publish that package for the runtime's distribution/architecture, the cell fails clearly and leaves the engine unavailable. Do not silently substitute another OpenFOAM distribution.


In [ ]:
ENABLE_OPENFOAM_INSTALL = False  # Set True only after reviewing the official installer and package.
OPENFOAM_PACKAGE = 'openfoam2606-default'
OPENFOAM_REPO_HELPER = 'https://dl.openfoam.com/add-debian-repo.sh'
openfoam_install = {'requested': ENABLE_OPENFOAM_INSTALL, 'package': OPENFOAM_PACKAGE, 'installed': False}
if ENABLE_OPENFOAM_INSTALL:
    helper = '/tmp/add-openfoam-repo.sh'
    fetch = subprocess.run(['curl', '-fsSL', OPENFOAM_REPO_HELPER, '-o', helper], capture_output=True, text=True, timeout=60)
    if fetch.returncode != 0:
        raise RuntimeError('Could not download official OpenFOAM repository helper: ' + fetch.stderr[-1000:])
    # The downloaded helper is from the official OpenFOAM domain; execution is explicit opt-in above.
    repo = subprocess.run(['bash', helper], capture_output=True, text=True, timeout=180)
    if repo.returncode != 0:
        raise RuntimeError('OpenFOAM repository setup failed: ' + repo.stderr[-2000:])
    update = subprocess.run(['apt-get', 'update'], capture_output=True, text=True, timeout=300)
    if update.returncode != 0:
        raise RuntimeError('apt update failed: ' + update.stderr[-2000:])
    install = subprocess.run(['apt-get', 'install', '-y', OPENFOAM_PACKAGE], capture_output=True, text=True, timeout=1200)
    if install.returncode != 0:
        raise RuntimeError('Pinned OpenFOAM package install failed; no alternate distribution was substituted: ' + install.stderr[-2000:])
    # Find the installed environment script without assuming one distro-specific path.
    found = subprocess.run(['bash', '-lc', "find /usr /opt -type f -path '*/etc/bashrc' 2>/dev/null | grep -E 'OpenFOAM-v2606|openfoam2606' | head -1"], capture_output=True, text=True, timeout=60)
    bashrc = found.stdout.strip()
    if not bashrc:
        raise RuntimeError('Package installed but expected v2606 bashrc was not found; mark backend unavailable and inspect package layout.')
    version = subprocess.run(['bash', '-lc', f'source {bashrc!r} && foamVersion'], capture_output=True, text=True, timeout=60)
    if version.returncode != 0:
        raise RuntimeError('OpenFOAM version probe failed: ' + version.stderr[-1000:])
    openfoam_install.update({'installed': True, 'bashrc': bashrc, 'version_output': version.stdout.strip(), 'package': OPENFOAM_PACKAGE})
else:
    print('OpenFOAM install is disabled. Set ENABLE_OPENFOAM_INSTALL=True after reviewing official package instructions.')
print(json.dumps(openfoam_install, indent=2))


### Optional OpenFOAM solver smoke run

After a successful installation, this opt-in cell runs the official `pitzDaily` simpleFoam tutorial in a copied case directory. It is a solver invocation smoke test, not a validation benchmark and not a GPU CFD test. It returns `EXECUTED_UNVERIFIED` semantics. The run is bounded by a wall-clock timeout.


In [ ]:
ENABLE_OPENFOAM_PITZDAILY_SMOKE = False  # Set True only after installation and after reviewing the case.
openfoam_smoke = {'requested': ENABLE_OPENFOAM_PITZDAILY_SMOKE, 'execution_state': 'NOT_REQUESTED'}
if ENABLE_OPENFOAM_PITZDAILY_SMOKE:
    if not openfoam_install.get('installed'):
        raise RuntimeError('OpenFOAM smoke run requested but the pinned v2606 install did not complete.')
    env_probe = subprocess.run(['bash', '-lc', f'source {openfoam_install["bashrc"]!r} && printf %s \"$FOAM_TUTORIALS\"'], capture_output=True, text=True, timeout=30)
    tutorial_root = env_probe.stdout.strip()
    source_case = os.path.join(tutorial_root, 'incompressible', 'simpleFoam', 'pitzDaily')
    if env_probe.returncode != 0 or not os.path.isdir(source_case):
        raise RuntimeError('Official pitzDaily tutorial not found in the installed tutorial tree: ' + source_case)
    import shutil as _shutil
    case_dir = '/content/pap-openfoam-pitzDaily'
    if os.path.exists(case_dir): _shutil.rmtree(case_dir)
    _shutil.copytree(source_case, case_dir)
    control = os.path.join(case_dir, 'system', 'controlDict')
    control_text = open(control, encoding='utf-8').read()
    import re as _re
    if not _re.search(r'\bendTime\s+\S+\s*;', control_text):
        raise RuntimeError('pitzDaily controlDict has no endTime field; refusing to edit an unknown case schema.')
    control_text = _re.sub(r'\bendTime\s+\S+\s*;', 'endTime 200;', control_text, count=1)
    open(control, 'w', encoding='utf-8').write(control_text)
    mesh_run = subprocess.run(['bash', '-lc', f'source {openfoam_install["bashrc"]!r} && blockMesh -case {case_dir!r}'], capture_output=True, text=True, timeout=300)
    if mesh_run.returncode != 0:
        raise RuntimeError('blockMesh failed: ' + mesh_run.stderr[-2000:])
    solve_run = subprocess.run(['bash', '-lc', f'source {openfoam_install["bashrc"]!r} && simpleFoam -case {case_dir!r}'], capture_output=True, text=True, timeout=900)
    if solve_run.returncode != 0:
        raise RuntimeError('simpleFoam failed or timed out: ' + solve_run.stderr[-2000:])
    openfoam_smoke = {'requested': True, 'execution_state': 'EXECUTED_UNVERIFIED', 'solver': 'simpleFoam', 'case': 'pitzDaily', 'case_directory': case_dir, 'max_iterations': 200, 'exit_code': solve_run.returncode, 'stdout_tail': solve_run.stdout[-4000:], 'stderr_tail': solve_run.stderr[-2000:], 'gpu_solver_claim': False, 'validation_state': 'UNVERIFIED'}
else:
    print('OpenFOAM smoke run is disabled. Set ENABLE_OPENFOAM_PITZDAILY_SMOKE=True after a successful pinned install.')
print(json.dumps(openfoam_smoke, indent=2))


## 7. Optional external backend activation policy

- **OpenFOAM:** install using the official distribution instructions for the selected release and Ubuntu base image. Do not mix OpenFOAM Foundation and OpenCFD packages in one environment. Pin release, repository and package checksum in the run manifest.
- **SU2:** use the official release/build instructions. Serial binaries and MPI-enabled builds have different capabilities. A binary installation must not be represented as a parallel build.
- **Code_Saturne:** use its supported build process. CUDA requires an explicit CUDA-enabled build, NVIDIA SDK and compatible numerical/MPI dependencies. A generic Colab GPU runtime is not proof that such a build is available.

Official installation references:
- OpenFOAM: https://www.openfoam.com/current-release
- SU2: https://su2code.github.io/docs_v7/Installation/
- Code_Saturne: https://code-saturne.org/doc/code_saturne-9.2/md__i_n_s_t_a_l_l.html

Colab resource policy and GPU availability: https://research.google.com/colaboratory/faq.html

## 8. Export a run manifest

This manifest is a proposed artifact. The host platform must bind it to actual canonical entity/configuration references and perform its own authorization, validation and evidence promotion. No notebook output should directly mark a design as physically verified.

In [ ]:
manifest = {
    "copyright": "Copyright © 1993-2026 Abhishek Choudhary. All rights reserved.",
    "notebook_id": "pap-cfd-backend-colab",
    "notebook_version": "1.0.0",
    "notebook_license": "GPL-3.0-only",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "runtime": runtime,
    "accelerator": GPU,
    "external_backend_probe": backend_probe,
    "external_backend_install": openfoam_install,
    "external_backend_smoke": openfoam_smoke,
    "gpu_solver_execution_claim": False,
    "reference_case": {
        "model": "STEADY_INCOMPRESSIBLE_LAMINAR_CHANNEL",
        "equation": "mu*d2u/dy2=dpdx",
        "parameters_si": {"height_m": H, "dynamic_viscosity_Pa_s": mu, "pressure_gradient_Pa_m": dpdx,
                           "bottom_wall_velocity_m_s": U0, "top_wall_velocity_m_s": U1},
        "flow_rate_per_span_m2_s": float(q_exact),
        "validation_state": "ANALYTICAL_REFERENCE_ONLY_NOT_PHYSICAL_VALIDATION"
    },
    "parameter_sweep": summary,
    "backend_execution": "NOT_EXECUTED_BY_THIS_NOTEBOOK",
    "trust_state": "UNVERIFIED",
    "subject_refs": [],
    "configuration_ref": None
}
manifest_bytes = json.dumps(manifest, indent=2, sort_keys=True).encode()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
with open("pap-cfd-colab-run-manifest.json", "wb") as f:
    f.write(manifest_bytes)
print("manifest sha256:", manifest_sha256)
print("wrote pap-cfd-colab-run-manifest.json")

## 9. Platform handoff

1. Download `pap-cfd-colab-run-manifest.json` and any solver outputs from the Colab Files pane.
2. Import them through the platform's explicit notebook-result adapter. Do not write directly to the canonical graph from the notebook.
3. Bind the run to the canonical subject IDs, configuration revision, mesh, solver release, case files, input checksums and output checksums.
4. Store stdout/stderr, solver exit status, convergence history, residual definitions, mesh statistics, resource usage and runtime metadata.
5. Classify the result as `EXECUTED_UNVERIFIED`. Independent benchmark comparison is required before `VERIFIED`; experimental correlation is required for physical validation claims.
6. Never promote notebook execution to HIL, PHIL or physical test evidence.

### Reproducibility warning

Colab virtual machines are ephemeral. Save source notebooks and generated artifacts to the repository or an explicitly authorized storage location. Do not rely on the runtime filesystem after disconnect.